# Provenance And Limitations Report Example

This notebook is a researcher-facing exploratory example for inspecting FungMod's provenance/limitation decision summary in the standard report workflow. It is report/output ergonomics, not an empirical validation, calibration, biology-evidence, or literature-comparison notebook. The example uses existing registry-backed virtual-experiment records, writes Markdown/HTML/index report artifacts, then inspects existing decision-support table links and rows.

In [ ]:
import os
from pathlib import Path

from fungal_model import virtual_experiment

OUTPUT_ROOT = Path(os.environ.get("FUNGMOD_NOTEBOOK_OUTPUT_ROOT", "outputs/notebooks"))
OUTPUT_DIR = OUTPUT_ROOT / "15_provenance_limitations_report_example"
REGISTRY = Path("data_registry/registry_index.yml")


Create a small public-API virtual experiment from researcher-facing names. This uses an existing supported registry-backed case so the notebook can focus on report usage and table inspection rather than implementing any scientific logic.

In [ ]:
study = virtual_experiment(
    fungi="beta-glucosidase source",
    substrates="cellobiose substrate",
    environments="30C_pH5_assay",
    registry=REGISTRY,
)

[(report.status, report.required_processes) for report in study.preflight(mode="exploratory")]


Run one exploratory sample and write the standard output bundle. `quicklook=False` keeps this example focused on existing CSV tables and report artifacts rather than presentation figures.

In [ ]:
result = study.simulate(
    mode="exploratory",
    n_samples=1,
    seed=23,
    output_dir=OUTPUT_DIR,
    quicklook=False,
)

result.write_summary()
result.write_manifest()
report_path = result.write_report(OUTPUT_DIR / "report", include_html=True, include_index=True)

report_path, sorted(path.name for path in OUTPUT_DIR.glob("*.csv"))


Inspect the provenance/limitation decision summary from the Markdown report. The section is derived only from existing standard output tables; it is not validation, calibration, empirical comparison, or inferred biology.

In [ ]:
report_text = report_path.read_text(encoding="utf-8")
summary_start = report_text.index("## Provenance and limitation decision summary")
summary_end = report_text.index("\n## Limitations", summary_start)
decision_summary = report_text[summary_start:summary_end]

assert "derived only from existing `assumption_summary.csv`" in decision_summary
assert "not validation, calibration, empirical comparison, or inferred biology" in decision_summary
assert "Source row counts: assumptions=" in decision_summary
assert "Limitation severity counts:" in decision_summary
assert "Assumption/provenance allowed-use labels present:" in decision_summary
assert "Exploratory-prior provenance rows:" in decision_summary

decision_summary.splitlines()[:8]


Load the decision-support tables through public result accessors. Missing-parameter and suggested-experiment rows may be empty for a supported exploratory case, but the table files and report links remain explicit.

In [ ]:
assumption_rows = result.assumption_summary()
limitation_rows = result.limitations()
provenance_rows = result.provenance()
missing_parameter_rows = result.missing_parameters()
suggested_experiment_rows = result.suggested_experiments()

assert assumption_rows
assert limitation_rows
assert provenance_rows
assert isinstance(missing_parameter_rows, list)
assert isinstance(suggested_experiment_rows, list)

{
    "assumptions": len(assumption_rows),
    "limitations": len(limitation_rows),
    "missing_parameters": len(missing_parameter_rows),
    "suggested_experiments": len(suggested_experiment_rows),
    "provenance": len(provenance_rows),
}


The optional HTML sidecar and report-folder index are navigation aids over existing artifacts. Check the decision-support table links before sharing the report so readers can inspect assumptions, limitations, missing inputs, suggested follow-up work, and provenance rows directly.

In [ ]:
report_dir = OUTPUT_DIR / "report"
assert (report_dir / "virtual_experiment_report.md").exists()
assert (report_dir / "virtual_experiment_report.html").exists()
assert (report_dir / "index.html").exists()

index_text = (report_dir / "index.html").read_text(encoding="utf-8")
html_text = (report_dir / "virtual_experiment_report.html").read_text(encoding="utf-8")
decision_support_files = (
    "assumption_summary.csv",
    "limitations_table.csv",
    "missing_parameters.csv",
    "suggested_experiments.csv",
    "provenance_table.csv",
)

for filename in decision_support_files:
    assert (OUTPUT_DIR / filename).exists()
    assert filename in index_text
    assert f"../{filename}" in html_text

decision_support_files
